In [1]:
"""
Train probes
"""
None

In [2]:
"""
Imports
"""
import torch
from datasets import load_dataset
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import gc
import pickle
import importlib
import cupy
import cuml
import sklearn
import yaml

from utils.memory import check_memory, clear_all_cuda_memory
from utils.loader import load_model_and_tokenizer, load_custom_forward_pass
from utils.probes import run_and_export_states, run_projections, check_max_seq_len

main_device = 'cuda:0'
seed = 123

clear_all_cuda_memory()
check_memory()

ws = '/workspace/prompt-injection-as-role-confusion'

All CUDA memory cleared on all devices.
Device 0: NVIDIA H200 NVL
  Allocated: 0.00 GB
  Reserved: 0.00 GB
  Total: 139.80 GB



# Load model

In [3]:
"""
Load the base tokenizer/model
"""
model_prefix = 'gptoss-20b'
tokenizer, model, model_architecture, model_n_layers = load_model_and_tokenizer(model_prefix, device = main_device)

check_memory()

You have loaded an FP4 model on CPU and have a CUDA/XPU device available, make sure to set your model on a GPU/XPU device in order to run your model. To remove this warning, pass device_map = 'cuda' or device_map = 'xpu'. 


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Expert precision: FloatType(bitwidth_exponent=2, bitwidth_mantissa=1, is_signed=True)
Attention implementation: kernels-community/vllm-flash-attn3
Device 0: NVIDIA H200 NVL
  Allocated: 12.85 GB
  Reserved: 12.90 GB
  Total: 139.80 GB



In [4]:
"""
Load custom forward pass and verify equality to base model forward pass
"""
run_forward_with_hs = load_custom_forward_pass(model_architecture, model, tokenizer)

LM loss: 3.984269618988037
Hidden states layers (pre-mlp | post-layer): 24 | 24
Hidden state size (pre-mlp | post-layer): torch.Size([1280, 2880]) | torch.Size([1280, 2880])
Verified custom forward pass successfully matches original model output!


In [5]:
"""
Test generation is sensible
"""
def test_generation():
    conv = tokenizer.apply_chat_template(
        [{"role": "user", "content": "Write a haiku about GPUs"},],
        tokenize = False,
        enable_thinking = True,
        add_generation_prompt = True
    )
    inputs = tokenizer(conv, return_tensors = 'pt')
    gen_ids = model.generate(inputs['input_ids'].to(main_device), max_new_tokens = 100, do_sample = False)
    print(tokenizer.batch_decode(gen_ids, skip_special_tokens = False)[0])    

test_generation()

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-09-04

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Write a haiku about GPUs<|end|><|start|>assistant<|channel|>analysis<|message|>The user wants a haiku about GPUs. Haiku is 5-7-5 syllable structure. Should be about GPUs. Let's craft a haiku. 5 syllables first line, 7 second, 5 third. Let's think: "Pixels dance, bright" maybe. Count syllables: "Pixel" 2, "dance" 1, "bright" 1 => 4. Need 5. Maybe "Pixels dance, bright glow" Count


# Train probes

## Prep dataset

In [6]:
"""
Load SFT dataset
"""
n_sample_size = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['n_sample_size']

def load_raw_ds():

    def get_c4():
        return load_dataset('allenai/c4', 'en', split = 'validation', streaming = True).shuffle(seed = seed, buffer_size = 50_000)
    
    def get_dolma3():
        return load_dataset('allenai/dolma3_mix-150B-1025', split = 'train', revision = '3a8349c', streaming = True).shuffle(seed = seed, buffer_size = 50_000)
    
    def get_data(ds, n_samples, data_source):
        raw_data = []
        ds_iter = iter(ds)
        for _ in range(n_samples):
            sample = next(ds_iter, None)
            if sample is None:
                break
            raw_data.append({'text': sample['text'], 'source': data_source})
        return raw_data
    
    return get_data(get_c4(), int(n_sample_size * .25), 'c4')  + get_data(get_dolma3(), int(n_sample_size * .75), 'dolma3')

raw_data = load_raw_ds()

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

In [7]:
"""
Test rendering
"""
import utils.role_templates
importlib.reload(utils.role_templates)
from utils.role_templates import render_single_message, render_mixed_cot

def test_render():
    print(tokenizer.apply_chat_template(
        [
            {'role': 'user', 'content': 'Hi! I am a dog and I like to bark'},
            {'role': 'assistant', 'content': 'Hello! What a lovely dog you are!'}
        ],
        tokenize = False, padding = 'max_length', truncation = True, max_length = 512, add_generation_prompt = True,
        enable_thinking = True
    ))
    print('--')
    print(render_single_message(model_prefix, role = 'user', content ='Hi'))
    print('--')
    print(render_mixed_cot(model_prefix, 'The user...', 'Yes!'))
    return True

test_render()

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-09-04

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hi! I am a dog and I like to bark<|end|><|start|>assistant<|channel|>final<|message|>Hello! What a lovely dog you are!<|end|><|start|>assistant
--
<|start|>user<|message|>Hi<|end|>
--
<|start|>assistant<|channel|>analysis<|message|>The user...<|end|><|start|>assistant<|channel|>final<|message|>Yes!<|end|>


True

In [8]:
"""
Validate forward passes work
"""
# Define opening texts to append at the start of each role for training. If multiple, will randomize equally.
train_prefixes = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['train_prefixes']
for p in train_prefixes:
    print(p)

@torch.no_grad()
def test1():
    conv = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': 'Hi stinky'}],
        tokenize = False,
        add_generation_prompt = True
    )
    inputs = tokenizer(conv, add_special_tokens = True, return_tensors = 'pt')
    gen_ids = model.generate(inputs['input_ids'].to(main_device), attention_mask = inputs['attention_mask'].to(main_device), max_new_tokens = 500, do_sample = False)
    df = pd.DataFrame({'token_id': gen_ids[0].tolist(), 'token': tokenizer.convert_ids_to_tokens(gen_ids[0].tolist()),})
    print(tokenizer.decode(df['token_id'].tolist(), skip_special_tokens = False))
    return df

@torch.no_grad()
def test2():
    """Quick test - can the model run forward passes correctly?"""
    for tr_prefix in train_prefixes:
        conv = tr_prefix + render_single_message(model_prefix, 'user', 'Where is Atlanta?') +\
            (
            '<|start|>assistant<|channel|>analysis<|message|>' # gpt-oss-*
            # '<|im_start|>assistant\n<think></think>' # Nemotron
            # '<|assistant|>\n<think>' # GLM-4.6V-Flash
            # '<|im_start|>assistant\n<think>\n' # Qwen3
            # '\n<|begin_assistant|>\nHere are my reasoning steps:\n' # Apriel-1.6-15B-Thinker
            # '<|assistant|><think>' # GLM-4.7-Flash
            )
        inputs = tokenizer(conv, add_special_tokens = False, return_tensors = 'pt')
        gen_ids = model.generate(inputs['input_ids'].to(main_device), attention_mask = inputs['attention_mask'].to(main_device), max_new_tokens = 500, do_sample = False)
        print(tokenizer.decode(gen_ids[0], skip_special_tokens = False))
        print('\n')

test1()
test2()

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2026-09-04

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Hi stinky<|end|><|start|>assistant<|channel|>analysis<|message|>The user says "Hi stinky". They might be addressing the assistant with a nickname. The assistant should respond politely. The user might be joking. The assistant should respond friendly. There's no request. Just greet. Probably respond with a friendly greeting.<|end|><|start|>assistant<|channel|>final<|message|>Hey there! How can I help you today?<|return|>


<|start|>user<|message|>Where is Atlanta?<|end|><|start|>assistant<|channel|>analysis<|message|>We need to answer: "Where is Atlanta?" The user asks location. We should respond with a concise answer: Atlanta is the capital and largest city of the U.S. state of Georgia, located in the north-central part of the state. It's in the southeastern United States. Provide coordinates maybe. Also mention it's in the U.S. Provide context.<|end|><|start|>assistant<|channel|>final<|message|>**Atlanta** is the capital and largest city of the U.S. state of **Georgia**.  
- **Location:** It sits in the north‑central part of Georgia, roughly 70 mi (110 km) south of the state’s border with South Carolina.  
- **Coordinates:** 33.7490° N latitude, 84.3880° W longitude.  
- **Region:** Part of the southeastern United States, within the larger Atlanta metropolitan area (the “Atlanta–Sandy Springs–Alpharetta” metro region).<|return|>




In [9]:
"""
Create sample sequences
"""
SEQLEN = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['seq_len']
NESTED_REASONING = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['nested_reasoning']
GENERALIZE_PREFIX = False

def get_sample_seqs_for_input_seq(probe_text, partner_text, prefix = ''):
    """
    Helper to convert a single input sequence into all examples

    Params
        @probe_text: The text we're extracting states from (appears in all roles)
        @partner_text: Random paired text (only used in merged sample's assistant position)
    """
    seqs = []

    gen_prefix = partner_text if GENERALIZE_PREFIX else ''

    # Only test system roles for gptoss-20b for time
    if model_prefix in ['gptoss-20b']:
        seqs.append({
            'role': 'system',
            'prompt': prefix + gen_prefix + render_single_message(model_prefix, role = 'system', content = probe_text)
        })

    for role in ['user', 'tool', 'cot']:
        seqs.append({
            'role': role,
            'prompt': prefix + gen_prefix + render_single_message(model_prefix, role = role, content = probe_text)
        })

    # If merge AND LRM, render with CoT prefix
    if NESTED_REASONING:
        seqs.append({
            'role': 'assistant',
            'prompt': prefix + render_mixed_cot(model_prefix, cot = partner_text, assistant = probe_text)
        })
    # Else render standalone
    else:
        seqs.append({
            'role': 'assistant',
            'prompt': prefix + gen_prefix + render_single_message(model_prefix, role = 'assistant', content = probe_text)
        })

    return seqs

def build_sample_seqs(train_prefixes):
    """
    Build all sample sequences
    """
    truncated_texts = tokenizer.batch_decode(tokenizer([t['text'] for t in raw_data], add_special_tokens = False, padding = False, truncation = True, max_length = SEQLEN).input_ids)
    n_seqs = len(truncated_texts)

    np.random.seed(seed)
    partner_lengths = ((np.random.beta(0.5, 4.0, size = n_seqs) * (SEQLEN/2 + 1)).astype(int)).tolist()
    partner_texts = [
        tokenizer.decode(tokenizer(text['text'], add_special_tokens = False, padding = False, truncation = True, max_length = int(partner_lengths[i])).input_ids)
        for i, text in enumerate(raw_data)
    ]
    perm = np.random.permutation(n_seqs)
    while np.any(perm == np.arange(n_seqs)):
        perm = np.random.permutation(n_seqs)

    sampled_prefixes = np.random.choice(train_prefixes, size = n_seqs)

    input_list = []
    for base_ix, base_text in enumerate(truncated_texts):
        partner_text = partner_texts[int(perm[base_ix])].strip()
        prefix = sampled_prefixes[base_ix]

        for seq in get_sample_seqs_for_input_seq(base_text, partner_text, prefix):
            row = {'question_ix': base_ix, 'question': base_text, **seq}
            input_list.append(row)

    input_df = pd.DataFrame(input_list).assign(prompt_ix = lambda df: list(range(len(df))))
    return input_df

input_df = build_sample_seqs(train_prefixes = train_prefixes)
display(input_df)

for p in [row['prompt'] for row in input_df.pipe(lambda df: df[df['question_ix'] == 2]).to_dict('records')]:
    print(p)
    print("=" * 80)

,question_ix,question,role,prompt,prompt_ix
0,0,Thank you for your interest in Broadway Grand ...,system,<|start|>system<|message|>Thank you for your i...,0
1,0,Thank you for your interest in Broadway Grand ...,user,<|start|>user<|message|>Thank you for your int...,1
2,0,Thank you for your interest in Broadway Grand ...,tool,<|start|>functions. to=assistant<|channel|>com...,2
3,0,Thank you for your interest in Broadway Grand ...,cot,<|start|>assistant<|channel|>analysis<|message...,3
4,0,Thank you for your interest in Broadway Grand ...,assistant,<|start|>assistant<|channel|>final<|message|>T...,4
...,...,...,...,...,...
1240,248,C. W. Stanford Auditorium Addition\n\nOrange C...,system,<|start|>system<|message|>C. W. Stanford Audit...,1240
1241,248,C. W. Stanford Auditorium Addition\n\nOrange C...,user,<|start|>user<|message|>C. W. Stanford Auditor...,1241
1242,248,C. W. Stanford Auditorium Addition\n\nOrange C...,tool,<|start|>functions. to=assistant<|channel|>com...,1242
1243,248,C. W. Stanford Auditorium Addition\n\nOrange C...,cot,<|start|>assistant<|channel|>analysis<|message...,1243


<|start|>system<|message|>Kess V2 CPU Repair Chip work For China Kess V2 Master Clone. Kess V2 CPU NXP fix chip With 60 Tokens. Kess V2 CPU Repair Chip Without J-Link JLINK V8+ ARM USB-JTAG Adapter Emulator.
Note: Please use this Kess V2 CPU chip to change the broken chip when you update your KESS V2.
1) You update your KESS V2.
2) Your tokens are run out.
For old version KESS V2, the tokens are limited about 64 times. If it prompt "you have last * program time", then it is time for you to change this chip. This chip has about 60 tokens, usually for ordinary ECU, when you use KESS V2 to read and write, it does not need tokens, only for special models, it needs tokens when use.
Which chip you need to change?<|end|>
<|start|>user<|message|>Kess V2 CPU Repair Chip work For China Kess V2 Master Clone. Kess V2 CPU NXP fix chip With 60 Tokens. Kess V2 CPU Repair Chip Without J-Link JLINK V8+ ARM USB-JTAG Adapter Emulator.
Note: Please use this Kess V2 CPU chip to change the broken chip when 

In [10]:
""" 
Load dataset into a dataloader which returns original tokens - important for BPE tokenizers to reconstruct the correct string later
"""
from utils.dataset import ReconstructableTextDataset, stack_collate
from torch.utils.data import DataLoader

max_seqlen = check_max_seq_len(tokenizer, input_df['prompt'].tolist())
train_dl = DataLoader(
    ReconstructableTextDataset(input_df['prompt'].tolist(), tokenizer, max_length = max_seqlen, prompt_ix = input_df['prompt_ix'].tolist()),
    batch_size = 32,
    shuffle = False,
    collate_fn = stack_collate
)

## Get hidden states

In [11]:
"""
Run forward passes
"""
layers_to_probe = list(range(0, model_n_layers, 4)) if model_n_layers >= 30 else list(range(0, model_n_layers, 2))
res = run_and_export_states(model, tokenizer, run_model_return_states = run_forward_with_hs, dl = train_dl, layers_to_keep_acts = layers_to_probe)

# Convert to f16 for cupy compatability
all_probe_hs = res['all_hs'].to(torch.float16)
all_probe_hs = {layer_ix: all_probe_hs[:, save_ix, :] for save_ix, layer_ix in enumerate(layers_to_probe)}
del res['all_hs']
gc.collect()

  0%|          | 0/39 [00:00<?, ?it/s]

---------
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|

  3%|▎         | 1/39 [00:04<03:08,  4.95s/it]

  5%|▌         | 2/39 [00:09<03:00,  4.89s/it]

  8%|▊         | 3/39 [00:14<02:53,  4.83s/it]

...

 95%|█████████▍| 37/39 [02:59<00:09,  4.90s/it]

 97%|█████████▋| 38/39 [03:04<00:04,  4.93s/it]

100%|██████████| 39/39 [03:10<00:00,  5.19s/it]

100%|██████████| 39/39 [03:10<00:00,  4.88s/it]

6554

## Label roles


In [12]:
"""
Take each token and label it with its correct role (user/assistant/system/tool/cot); check the counts are as expected
"""
import utils.role_assignments
importlib.reload(utils.role_assignments)
from utils.role_assignments import label_content_roles
from utils.substring_assignments import flag_message_types

probe_sample_df = (
    # Flag all roles
    label_content_roles(model_prefix, res['sample_df'])    
    .assign(sample_ix = lambda df: range(0, len(df)))
    # Flag those that match target role (for retain later)
    .merge(input_df[['prompt_ix', 'role']].rename(columns = {'role': 'target_role'}), how = 'inner', on = 'prompt_ix')
    .assign(match_target_role = lambda df: np.where(df['role'] == df['target_role'], True, False))
    # Flag those that match the prepend (for drop later)
    .pipe(lambda df: flag_message_types(df, train_prefixes, allow_ambiguous = True))
    .drop(columns = 'base_message')
    # Drop prepend text + non-content tags
    .pipe(lambda df: df[(df['base_message_ix'].isna()) & (df['is_content'] == True) & (df['role'].notna()) & (df['match_target_role'])])
)

# Verify role counts are accurate (should be exactly equal for most models, except when models where role tags can be merged with content into single toks)
display(probe_sample_df.groupby('role', as_index = False).agg(count = ('sample_ix', 'count')))

# Validate roles are flagged correctly
display(
    probe_sample_df\
    .pipe(lambda df: df[df['prompt_ix'] <= 14])\
    .groupby(['prompt_ix', 'seg_ix', 'role'], as_index = False)\
    .agg(combined_text = ('token', ''.join))\
    .assign(eot = lambda df: df['combined_text'].str[-30:])
)

,role,count
0,assistant,141078
1,cot,141078
2,system,141078
3,tool,141078
4,user,141078


,prompt_ix,seg_ix,role,combined_text,eot
0,0,0,system,Thank you for your interest in Broadway Grand ...,"5, or by fax at (616)235-6282."
1,1,0,user,Thank you for your interest in Broadway Grand ...,"5, or by fax at (616)235-6282."
2,2,0,tool,Thank you for your interest in Broadway Grand ...,"5, or by fax at (616)235-6282."
3,3,0,cot,Thank you for your interest in Broadway Grand ...,"5, or by fax at (616)235-6282."
4,4,0,assistant,Thank you for your interest in Broadway Grand ...,"5, or by fax at (616)235-6282."
5,5,0,system,Do you need your site to be found for a specif...,livery capabilities worldwide.
6,6,0,user,Do you need your site to be found for a specif...,livery capabilities worldwide.
7,7,0,tool,Do you need your site to be found for a specif...,livery capabilities worldwide.
8,8,0,cot,Do you need your site to be found for a specif...,livery capabilities worldwide.
9,9,0,assistant,Do you need your site to be found for a specif...,livery capabilities worldwide.


## Run probes

In [13]:
"""
First run a quick grid search over hyperparmeters
"""
SKIP_FIRST_N = 32 if NESTED_REASONING else 0

def fit_lr(x_train, y_train, x_test, y_test, add_scaling = False, **lr_params):
    """
    Fit a probe
    """
    steps = []
    if add_scaling:
        steps.append(('scaler', cuml.preprocessing.StandardScaler()))
    steps.append(('clf', cuml.linear_model.LogisticRegression(penalty = 'l2', max_iter = 5_000, linesearch_max_iter = 100, fit_intercept = True, **lr_params)))
    lr_model = sklearn.pipeline.Pipeline(steps)
    lr_model.fit(x_train, y_train)
    accuracy = lr_model.score(x_test, y_test)
    y_test_pred = lr_model.predict(x_test)
    y_test_prob = lr_model.predict_proba(x_test)
    nll = cuml.metrics.log_loss(y_test, y_test_prob,)
    return lr_model, accuracy, nll, y_test_pred

def get_probe_result(sample_df, layer_hs, roles_map, add_scaling = False, **lr_params):
    """
    Get probe results for a single layer and label combination

    Params:
        @sample_df: The sample-level df; with a column `sample_ix` indicating the token order of 0...T-1;
            the actual df may be shorter due to pre-filters
        @layer_hs: A tensor of probe hidden states for a layer, of T x D
        @roles_map: The mapping order of the roles; a dict {}

    Description:
        Trains only on content space for given roles
    """
    # Train/test split
    prompt_ix_train, prompt_ix_test = cuml.train_test_split(sample_df['prompt_ix'].unique(), test_size = 0.1, random_state = seed)
    train_df = sample_df[sample_df['prompt_ix'].isin(prompt_ix_train)]
    test_df = sample_df[sample_df['prompt_ix'].isin(prompt_ix_test)]

    # Get y labels
    role_labels_train_cp = cupy.asarray([roles_map[r] for r in train_df['role']])
    role_labels_test_cp = cupy.asarray([roles_map[r] for r in test_df['role']])

    # Get x labels
    x_train_cp = cupy.asarray(layer_hs[train_df['sample_ix'].tolist(), :].to(torch.float32).detach().cpu())
    x_test_cp = cupy.asarray(layer_hs[test_df['sample_ix'].tolist(), :].to(torch.float32).detach().cpu())
    
    if (len(train_df) != x_train_cp.shape[0]):
        raise Exception(f"Shape mismatch!")

    uniq_train = np.unique(role_labels_train_cp.get())
    if len(uniq_train) < len(roles_map):
        raise Exception(f"Skipping mapping {roles_map}: missing roles in train", uniq_train)

    lr_model, test_acc, test_nll, y_test_pred = fit_lr(
        x_train_cp, role_labels_train_cp, x_test_cp, role_labels_test_cp,
        add_scaling = add_scaling,
        **lr_params
    )

    # Classification metrics
    results_df =\
        test_df\
        .assign(pred = y_test_pred.tolist())\
        .assign(pred = lambda df: df['pred'].map({v: k for k, v in roles_map.items()}))\
        .assign(is_acc = lambda df: df['role'] == df['pred'])

    acc_by_role =\
        results_df\
        .groupby(['role', 'pred'], as_index = False)\
        .agg(count = ('sample_ix', 'count'))

    acc_by_pos =\
        results_df\
        .groupby('token_in_seg_ix', as_index = False)\
        .agg(count = ('sample_ix', 'count'), acc = ('is_acc', 'mean'))

    return {
        'probe': lr_model, 'acc': test_acc, 'nll': test_nll,
        'acc_by_role': acc_by_role, 'acc_by_pos': acc_by_pos
    }

def test_c(C, add_scaling):
    """
    Test hyperparams: mid-layer, roles for UCAT rolespace
    """
    test_roles = ['user', 'assistant', 'tool']
    test_layer_ix = layers_to_probe[(len(layers_to_probe) - 1) // 2] 
    probe_res = get_probe_result(
        sample_df = probe_sample_df[(probe_sample_df['role'].isin(test_roles)) & (probe_sample_df['token_in_seg_ix'] >= SKIP_FIRST_N)].reset_index(drop = True),
        layer_hs = all_probe_hs[test_layer_ix],
        roles_map = {x: i for i, x in enumerate(test_roles)},
        add_scaling = add_scaling,
        C = C
    )
    return {'C': C, 'add_scaling': add_scaling, 'val_acc': probe_res['acc'], 'val_nll': probe_res['nll']}

clear_all_cuda_memory()
gc.collect()

# for scale_val in [False]:
#     print(f"Scaling: {scale_val}")
#     display(pd.DataFrame([test_c(c_val, add_scaling = scale_val) for c_val in tqdm([1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2])]))

All CUDA memory cleared on all devices.


0

In [14]:
"""
Run role probes at every fourth layer for role combinations specified in all_role_combinations (excl CoT role for non-reasoning models)
"""
train_params = yaml.safe_load(open('config/probe.yaml'))[model_prefix]['train_params']

all_role_combinations = [
    ('user', 'assistant'),
    ('user', 'assistant', 'tool'),
]

if model_prefix in ['gptoss-20b']:
    all_role_combinations += [
        ('user', 'cot', 'assistant'),
        ('user', 'cot', 'assistant', 'tool'),
        ('system', 'user', 'assistant'),
        ('system', 'user', 'assistant', 'tool'),
        ('system', 'user', 'cot', 'assistant'),
        ('system', 'user', 'cot', 'assistant', 'tool'),
    ]

all_role_combinations = [
    {
        'roles': list(roles),
        'roles_map': {x: i for i, x in enumerate(roles)},
        # Sample df for only those roles - filtering here is fine since we retain sample_ix which get_probe_result() uses to trace the original token
        'sample_df': probe_sample_df.pipe(lambda df: df[(df['role'].isin(roles)) & (df['token_in_seg_ix'] >= SKIP_FIRST_N)]).reset_index(drop = True),
    }
    for roles in all_role_combinations
]

clear_all_cuda_memory()
gc.collect()

all_probes = []
for roles_dict in all_role_combinations:
    print(f"Training {roles_dict['roles']}")
    for layer_ix in layers_to_probe:
        probe_res = get_probe_result(
            sample_df = roles_dict['sample_df'],
            layer_hs = all_probe_hs[layer_ix],
            roles_map = roles_dict['roles_map'],
            add_scaling = train_params['add_scaling'],
            C = train_params['C']
        )
        print(f"  Layer [{layer_ix}]: {probe_res['acc']:.2f}")

        all_probes.append({
            **probe_res,
            'layer_ix': layer_ix,
            'role_space': roles_dict['roles'],
            'roles_map': roles_dict['roles_map'],
            'n_inputs': len(roles_dict['sample_df'])
        })

print(f"Num probes: {str(len(all_probes))}")
print(f"Probe layers:\n  {', '.join([str(x) for x in layers_to_probe])}")

All CUDA memory cleared on all devices.
Training ['user', 'assistant']


[2026-09-04 17:08:30.829] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step


  Layer [0]: 0.46


[2026-09-04 17:08:35.183] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [2]: 0.71


[2026-09-04 17:08:36.950] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [4]: 0.65


...

[2026-09-04 17:20:58.118] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [14]: 0.64


[2026-09-04 17:21:08.887] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [16]: 0.72


[2026-09-04 17:21:22.506] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [18]: 0.71


[2026-09-04 17:21:32.579] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [20]: 0.67


[2026-09-04 17:21:50.241] [CUML] [warning] L-BFGS line search failed (code 1); stopping at the last valid step
  Layer [22]: 0.67
Num probes: 96
Probe layers:
  0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22


In [15]:
"""
Save probes + metrics
"""
def validate_accuracy_and_save(all_probes):
    """
    Calculate accurcay by (rolespace, layer, role) and save
    """
    print('Val accuracy by layer:')
    display(
        pd.DataFrame(all_probes)[['layer_ix', 'role_space', 'acc']]\
            .assign(acc = lambda df: df['acc'].round(2), role_space = lambda df: df['role_space'].apply(lambda x: ','.join(([r[0] for r in x]))))\
            .pivot(index = 'layer_ix', columns = 'role_space', values = 'acc')
    )

    print('Concatenating across layers and saving...')
    acc_by_role = pd.concat([p['acc_by_role'].assign(model = model_prefix, layer_ix = p['layer_ix'], role_space = ','.join(p['role_space'])) for p in all_probes], ignore_index = True)
    acc_by_pos =\
        pd.concat([p['acc_by_pos'].assign(model = model_prefix, layer_ix = p['layer_ix'], role_space = ','.join(p['role_space'])) for p in all_probes], ignore_index = True)\
        .assign(acc = lambda df: df['acc'].round(4))

    acc_by_role.to_csv(f'{ws}/experiments/role-analysis/outputs/probe-training/acc_by_role_{model_prefix}.csv', index = False)
    acc_by_pos.to_csv(f'{ws}/experiments/role-analysis/outputs/probe-training/acc_by_pos_{model_prefix}.csv', index = False)

    with open(f'{ws}/experiments/role-analysis/outputs/probes/{model_prefix}.pkl', 'wb') as f:
        pickle.dump(all_probes, f)

    print('Accuracy by role:')
    base_sums = acc_by_role.groupby(['role', 'layer_ix', 'role_space'], as_index = False).agg(base_sum = ('count', 'sum'))

    acc_by_role =\
        acc_by_role\
        .pipe(lambda df: df[df['role'] == df['pred']])\
        .groupby(['role', 'layer_ix', 'role_space'], as_index = False)\
        .agg(sum = ('count', 'sum'))\
        .merge(base_sums, on = ['layer_ix', 'role_space', 'role'], how = 'inner')\
        .assign(acc = lambda df: df['sum']/df['base_sum'])\
        .pivot(index = ['role_space', 'layer_ix'], columns = 'role', values = 'acc')\
        .round(2)

    display(acc_by_role)

    return True

validate_accuracy_and_save(all_probes)

Val accuracy by layer:


role_space,"s,u,a","s,u,a,t","s,u,c,a","s,u,c,a,t","u,a","u,a,t","u,c,a","u,c,a,t"
layer_ix,,,,,,,,
0,0.23,0.19,0.16,0.14,0.46,0.29,0.25,0.19
2,0.35,0.36,0.26,0.28,0.71,0.51,0.43,0.36
4,0.40,0.38,0.31,0.30,0.65,0.54,0.47,0.38
6,0.59,0.63,0.59,0.51,0.85,0.74,0.69,0.61
8,0.73,0.82,0.81,0.70,0.92,0.88,0.85,0.80
10,0.69,0.76,0.74,0.62,0.89,0.82,0.79,0.72
12,0.70,0.72,0.70,0.59,0.88,0.80,0.77,0.67
14,0.77,0.79,0.80,0.64,0.92,0.85,0.83,0.75
16,0.88,0.87,0.88,0.72,0.97,0.91,0.90,0.82


Concatenating across layers and saving...


Accuracy by role:


role                              assistant   cot  system  tool  user
role_space              layer_ix                                     
system,user,assistant   0              0.39   NaN    0.10   NaN  0.25
                        2              0.61   NaN    0.20   NaN  0.29
                        4              0.61   NaN    0.26   NaN  0.38
                        6              0.78   NaN    0.47   NaN  0.59
                        8              0.83   NaN    0.68   NaN  0.71
...                                     ...   ...     ...   ...   ...
user,cot,assistant,tool 14             0.89  0.64     NaN  0.61  0.84
                        16             0.94  0.73     NaN  0.64  0.96
                        18             0.92  0.65     NaN  0.63  0.96
                        20             0.91  0.58     NaN  0.58  0.94
                        22             0.91  0.59     NaN  0.61  0.92

[96 rows x 5 columns]

True

In [16]:
"""
Portable sklearn clone of the cuML probes (cuML estimators will not unpickle off Hopper)
"""
from sklearn.linear_model import LogisticRegression
import numpy as np, pickle, pathlib

def _to_numpy(a):
    """cuML exposes coef_/intercept_ as cupy arrays, which refuse implicit numpy conversion."""
    return a.get() if hasattr(a, 'get') else np.asarray(a)

out = []
for p in all_probes:
    pipe = p["probe"]
    clf = pipe.named_steps["clf"] if hasattr(pipe, "named_steps") else pipe
    if not hasattr(pipe, "named_steps"):
        print("no named_steps on", type(p["probe"]))
    coef = _to_numpy(clf.coef_)
    intercept = _to_numpy(clf.intercept_)
    sk = LogisticRegression(C=5.0e-3, fit_intercept=True, max_iter=5000)
    n_rows, n_features = coef.shape
    # Binary problems carry a single coefficient row for two classes
    n_classes = 2 if n_rows == 1 else n_rows
    sk.classes_ = np.arange(n_classes)
    sk.coef_ = coef.astype(np.float64)
    sk.intercept_ = intercept.astype(np.float64)
    sk.n_features_in_ = n_features
    out.append({
        "probe": sk,
        "acc": p["acc"],
        "nll": p["nll"],
        "layer_ix": int(p["layer_ix"]),
        "role_space": list(p["role_space"]),
        "roles_map": dict(p["roles_map"]),
        "n_inputs": p.get("n_inputs"),
        "C": 5.0e-3,
        "feature": "pre_mlp",
    })

# Confirm the clone reproduces cuML probabilities on real activations
print("Verifying clones against cuML on real activations...")
max_diff = 0.0
for p, q in zip(all_probes, out):
    rs = q["role_space"]
    df = probe_sample_df[(probe_sample_df['role'].isin(rs)) & (probe_sample_df['token_in_seg_ix'] >= SKIP_FIRST_N)]
    ix = df['sample_ix'].tolist()[:2000]
    x = all_probe_hs[q["layer_ix"]][ix, :].to(torch.float32).detach().cpu()
    p_cuml = cupy.asnumpy(p["probe"].predict_proba(cupy.asarray(x)))
    p_sk = q["probe"].predict_proba(np.asarray(x, dtype=np.float64))
    max_diff = max(max_diff, float(np.abs(p_cuml - p_sk).max()))
print(f"max |p_cuml - p_sklearn| over all {len(out)} probes = {max_diff:.3e}")

dest = pathlib.Path(ws) / "experiments/role-analysis/outputs/probes/role_probes.pkl"
with open(dest, "wb") as f:
    pickle.dump(out, f)

five = [p for p in out if p["role_space"] == ["system", "user", "cot", "assistant", "tool"]]
assert {p["layer_ix"] for p in five} == set(range(0, 24, 2)), {p["layer_ix"] for p in five}
assert all(p["probe"].coef_.shape[0] == 5 for p in five)
print("wrote", dest, "n=", len(out), "five-role layers", sorted(p["layer_ix"] for p in five))
for p in five:
    print(f"  L{p['layer_ix']:02d} acc={p['acc']:.3f} coef={p['probe'].coef_.shape}")


Verifying clones against cuML on real activations...


max |p_cuml - p_sklearn| over all 96 probes = 1.908e-06
wrote /workspace/prompt-injection-as-role-confusion/experiments/role-analysis/outputs/probes/role_probes.pkl n= 96 five-role layers [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]
  L00 acc=0.143 coef=(5, 2880)
  L02 acc=0.276 coef=(5, 2880)
  L04 acc=0.298 coef=(5, 2880)
  L06 acc=0.510 coef=(5, 2880)
  L08 acc=0.704 coef=(5, 2880)
  L10 acc=0.621 coef=(5, 2880)
  L12 acc=0.595 coef=(5, 2880)
  L14 acc=0.638 coef=(5, 2880)
  L16 acc=0.725 coef=(5, 2880)
  L18 acc=0.710 coef=(5, 2880)
  L20 acc=0.667 coef=(5, 2880)
  L22 acc=0.674 coef=(5, 2880)
